In [ ]:
#   新版本，带上random sort

import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from matplotlib.colors import LinearSegmentedColormap
from scipy import interpolate

# 指定文件夹路径
folder_path = '/home/linkco/exa/Useful-Embedding/data/analyze/'

# 获取所有文件和文件夹名称
files = os.listdir(folder_path)

# 筛选出以.json结尾的文件
json_files = [f for f in files if f.endswith('.json')]

for file in json_files:
# for file in ['stella_en_400M_v5.json']:
    model_name = file.replace(".json", "")
    print(model_name)
    # 读取文件
    file_path = folder_path + "{}.json".format(model_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 提取任务信息
    tasks = data["task_name"]

    # 存放绘图对象
    plots = {}

    # 创建自定义颜色映射
    def create_color_gradient(base_color, n_colors):
        """创建从浅到深的颜色渐变"""
        base_rgb = plt.cm.colors.to_rgb(base_color)
        colors = [(*base_rgb, alpha) for alpha in np.linspace(0.3, 1, n_colors)]
        return colors
    

    fused_heatmap_list = []
    for task_name, task_data in tasks.items():


        baseline = task_data["defult_score"]
        
        # 捋顺数据
        
        task_data["random_score"] = {k: task_data["random_score"][k] for k in sorted(task_data["random_score"].keys(), key=int)}
        task_data["sort_score"] = {k: task_data["sort_score"][k] for k in sorted(task_data["sort_score"].keys(), key=int)}
        # 收集 head/end scores
        records = []
        heatmaps = {}
        
        for split_size, split_data in task_data["split_win_size"].items():
            if not split_data:  # 跳过空的
                continue
            
            # 记录 chunk_win_size scores
            if "chunk_win_size" in split_data:
                for chunk_size, score_dict in split_data["chunk_win_size"].items():
                    records.append({
                        "split_win_size": int(split_size),
                        "chunk_win_size": int(chunk_size),
                        "random_score": score_dict["random_score"]["main_score"],
                        "head_score": score_dict["head_score"]["main_score"],
                        "end_score": score_dict["end_score"]["main_score"],
                        "sort_score": score_dict["sort_score"]["main_score"]
                    })
            
            # 记录 chunk_result
            if "chunk_result" in split_data:
                heatmaps[int(split_size)] = split_data["chunk_result"]
        
        # 转换为 DataFrame
        df_scores = pd.DataFrame(records)
        
        # 获取唯一的 split_win_size 值并排序
        split_sizes = sorted(df_scores["split_win_size"].unique())
        n_splits = len(split_sizes)
        
        # 创建颜色渐变

        # 定义Score Types的颜色和标签
        score_colors = {
            'head_score': '#1f77b4',    # 蓝色 - Best Chunks
            'end_score': '#ff7f0e',     # 橙色 - Poor Chunks  
            'random_score': '#2ca02c',  # 绿色 - Random Chunks
            'sort_score': '#000000'     # 红色 - Sort Chunks
        }

        blue_colors = create_color_gradient("#1f77b4", n_splits)
        orange_colors = create_color_gradient("#ff7f0e", n_splits)
        green_colors = create_color_gradient("#2ca02c", n_splits)
        black_colors = create_color_gradient("#000000", n_splits)
        
        # 绘制折线图
        plt.figure(figsize=(8, 6))
        
        # 绘制 head_score 和 end_score
        for i, split_size in enumerate(split_sizes):
            subset = df_scores[df_scores["split_win_size"] == split_size]
            
            # head_score - 蓝色系，颜色深浅表示 split_size
            plt.plot(subset["chunk_win_size"], subset["head_score"],
                    color=blue_colors[i], marker='o', linestyle='-',
                    label=f"head_score (split={split_size})" if i == 0 else "")
            
            # end_score - 橙色系，颜色深浅表示 split_size
            plt.plot(subset["chunk_win_size"], subset["end_score"],
                    color=orange_colors[i], marker='s', linestyle='--',
                    label=f"end_score (split={split_size})" if i == 0 else "")
            
            if i == 0:
                # sort_score - 黑色系，颜色深浅表示 split_size
                plt.plot(subset["chunk_win_size"], subset["sort_score"],
                        color="black", marker='^', linestyle=':',
                        label=f"sort_score" if i == 0 else "")
                
                # random_score - 橙色系，颜色深浅表示 split_size
                plt.plot(subset["chunk_win_size"], subset["random_score"],
                        color="green", marker='^', linestyle=':',
                        label=f"random_score" if i == 0 else "")
                
        
        # # 绘制 sort_score (使用黑色)
        # # 注意：sort_score 可能不随 split_size 变化，所以只绘制一次
        # plt.plot(df_scores["chunk_win_size"].unique(), 
        #         df_scores.groupby("chunk_win_size")["sort_score"].first(),
        #         color="black", marker='^', linestyle=':', label="sort_score")
        
        # # 绘制 random_score (使用黑色)
        # # 注意：random_score 可能不随 split_size 变化，所以只绘制一次
        # plt.plot(df_scores["chunk_win_size"].unique(), 
        #         df_scores.groupby("chunk_win_size")["random_score"].first(),
        #         color="green", marker='^', linestyle=':', label="random_score")
        
        # 添加基线
        plt.axhline(y=baseline, color="red", linestyle="--", label="baseline")
        
        # ✅ 设置 x 轴为 log2
        plt.xscale("log", base=2)
        xticks = sorted(df_scores["chunk_win_size"].unique())
        # plt.xticks(xticks, [f"2^{int(np.log2(x))}" for x in xticks])
        plt.xticks(xticks, xticks)

        plt.title(f"{model_name} - {task_name} \n Scores by different Analyze-Size & Dimensional-Size")
        plt.xlabel("Dimensional Size")
        plt.ylabel("Score")
        
        # 创建自定义图例
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch

        # 创建两个图例
        # 第一个图例：线型说明
        legend_elements1 = [
            # baseline 图例
            Line2D([0], [0], color='red', linestyle='--', label='Full'),
            # head_score 图例
            Line2D([0], [0], color='#1f77b4', marker='o', linestyle='-', label='Best'),
            # end_score 图例
            Line2D([0], [0], color='#ff7f0e', marker='s', linestyle='--', label='Poor'),
            # sort_score 图例
            Line2D([0], [0], color='#000000', marker='^', linestyle=':', label='Sort'),
            # random_score 图例
            Line2D([0], [0], color='#2ca02c', marker='^', linestyle=':', label='Random')
        ]
        
        # 第二个图例：颜色深浅说明（展示所有split_size对应的颜色）
        legend_elements2 = []
        for i, split_size in enumerate(split_sizes):
            legend_elements2.append(
                Patch(facecolor=black_colors[i], edgecolor=black_colors[i], 
                    label=f'{split_size}')
            )
        
        # 创建两个图例
        legend1 = plt.legend(handles=legend_elements1, loc='lower left', bbox_to_anchor=(0.7, 0), title='Types', framealpha=0.6)
        legend2 = plt.legend(handles=legend_elements2, loc='lower left', bbox_to_anchor=(0.8, 0), title='Analyze Size', framealpha=0.6)
        
        # 添加第二个图例到图表中
        plt.gca().add_artist(legend1)
        
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        plots[f"{task_name}_line"] = plt.gcf()
        plt.close()
        
        # 绘制 chunk_result 热力图
        if heatmaps:
            # 找出所有split_size中最大的长度
            max_length = max(len(arr) for arr in heatmaps.values())
            
            # 创建归一化的融合矩阵
            normalized_heatmaps = {}
            for split_size, chunk_result in heatmaps.items():
                # 归一化处理
                arr = np.array(chunk_result)
                normalized_arr = (arr - np.min(arr)) / (np.max(arr) - np.min(arr) + 1e-8)  # 添加小值避免除零
                
                # 如果长度不足，使用插值扩展到最大长度
                if len(normalized_arr) < max_length:
                    x_old = np.linspace(0, 1, len(normalized_arr))
                    x_new = np.linspace(0, 1, max_length)
                    f = interpolate.interp1d(x_old, normalized_arr, kind='linear', fill_value='extrapolate')
                    normalized_arr = f(x_new)
                
                normalized_heatmaps[split_size] = normalized_arr

            # 创建融合热力图
            fused_heatmap = np.zeros(max_length)
            
            for split_size, normalized_arr in normalized_heatmaps.items():
                fused_heatmap += normalized_arr

            fused_heatmap /= len(normalized_heatmaps)

            # 保留当前任务的权重分布数据
            fused_heatmap_list.append(fused_heatmap)


            
            # 绘制单独的热力图（归一化后）
            for split_size, normalized_arr in normalized_heatmaps.items():
                plt.figure(figsize=(12, 3))
                result_matrix = [normalized_arr]  # 单行热力图
                sns.heatmap(result_matrix, cmap="YlGnBu", cbar=True, xticklabels=False, 
                        yticklabels=[f"split={split_size}"], vmin=0, vmax=1)
                plt.title(f"{task_name} - Normalized chunk_result heatmap (split_win_size={split_size})")
                plots[f"{task_name}_heatmap_norm_{split_size}"] = plt.gcf()
                plt.close()
            
            # 绘制融合热力图
            plt.figure(figsize=(12, 3))
            result_matrix = [fused_heatmap]  # 单行热力图
            sns.heatmap(result_matrix, cmap="YlGnBu", cbar=True, xticklabels=False, 
                    yticklabels=["Fused"], vmin=0, vmax=1)
            plt.title(f"{task_name} - Fused chunk_result heatmap (weighted by split_size)")
            plots[f"{task_name}_heatmap_fused"] = plt.gcf()
            plt.close()
    
    # 计算各个维度根据不同任务，是否存在权重差异的使用上的不同，方差
    # # 将四个数组按列堆叠，形状变为 (4, n)
    # stacked = np.vstack(np.array(fused_heatmap_list))

    # # 沿着 axis=0 计算每列的方差
    # variances = np.var(stacked, axis=0)

    # # 对所有位置的方差求平均
    # mean_variance = np.mean(variances)

    # # print("每个索引位置的方差:", variances)
    # print("平均方差:", mean_variance)

    # 计算各个维度根据不同任务，是否存在权重差异的使用上的不同，方差
    if fused_heatmap_list:
        # 找到最长的数组长度
        max_length = max(len(arr) for arr in fused_heatmap_list)
        
        # 将所有数组扩展到相同长度（使用0填充或插值）
        padded_arrays = []
        for arr in fused_heatmap_list:
            if len(arr) < max_length:
                # 使用0填充到最大长度
                padded_arr = np.pad(arr, (0, max_length - len(arr)), mode='constant', constant_values=0)
            else:
                padded_arr = arr
            padded_arrays.append(padded_arr)
        
        # 将数组堆叠成二维数组
        stacked = np.vstack(padded_arrays)
        
        # 沿着 axis=0 计算每列的方差
        variances = np.var(stacked, axis=0)
        
        # 对所有位置的方差求平均
        mean_variance = np.mean(variances)
        
        # print("每个索引位置的方差:", variances)
        print("平均方差:", mean_variance)
    else:
        print("没有可用的融合热力图数据")

    

    # 保存所有图表为单独图片
    output_dir = "/home/linkco/exa/Useful-Embedding/data/analyze/{}_visualizations".format(model_name)
    os.makedirs(output_dir, exist_ok=True)

    saved_files = []
    for name, fig in plots.items():
        save_path = output_dir
        if "line" in name:
            save_path += "/line"
        elif "fused" in name:
            save_path += "/fused"

        os.makedirs(save_path, exist_ok=True)
        file_path = os.path.join(save_path, f"{name}.png")
        fig.savefig(file_path, dpi=300, bbox_inches="tight")
        saved_files.append(file_path)

In [ ]:
# 绘制不同模型的拟合结果
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from matplotlib.colors import LinearSegmentedColormap
from scipy import interpolate
from scipy.interpolate import interp1d

# 指定文件夹路径
folder_path = '/home/linkco/exa/Useful-Embedding/data/analyze/'

# 获取所有文件和文件夹名称
files = os.listdir(folder_path)

# 筛选出以.json结尾的文件
json_files = [f for f in files if f.endswith('.json')]

# 选择要比较的模型（可以根据需要修改这个列表）
models_to_compare = ['stella_en_400M_v5-GEDI-epoch_3', 'jina-embeddings-v3']  # 示例模型名称

# 存储所有模型的数据
all_models_data = {}

# 选择需要分析的最小切割窗口
analyz_split_win_size = 2

# 读取所有模型数据
for file in json_files:
    model_name = file.replace(".json", "")
    
    # 如果指定了要比较的模型列表，则只处理这些模型
    if models_to_compare and model_name in models_to_compare:
        continue
        
    print(f"Processing {model_name}")
    
    # 读取文件
    file_path = folder_path + "{}.json".format(model_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    all_models_data[model_name] = data

# 提取任务信息（假设所有模型有相同的任务）
tasks = list(all_models_data[list(all_models_data.keys())[0]]["task_name"].keys())

# 定义Score Types的颜色和标签
score_colors = {
    'head_score': '#1f77b4',    # 蓝色 - Best Chunks
    'end_score': '#ff7f0e',     # 橙色 - Poor Chunks  
    'random_score': '#2ca02c',  # 绿色 - Random Chunks
    'sort_score': '#000000'     # 红色 - Sort Chunks
}

score_labels = {
    'head_score': 'Best Chunks',
    'end_score': 'Poor Chunks',
    'random_score': 'Random Chunks',
    'sort_score': 'Sort Chunks'
}

# 定义模型的marks和line样式组合
model_styles = {
    'markers': ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'X', 'd'],
    'linestyles': ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]
}

# 为每个任务创建综合图表
for task_name in tasks:
    plt.figure(figsize=(10, 8))
    
    # 存储所有模型的归一化数据用于拟合
    all_normalized_data = {
        'head_score': [],
        'end_score': [],
        'random_score': [],
        'sort_score': []
    }
    
    # 存储每个模型的random_min值用于归一化
    random_mins = {}
    
    # 首先收集所有模型的random_min值
    for model_name, data in all_models_data.items():
        task_data = data["task_name"][task_name]
        baseline = task_data["defult_score"]
        
        # 获取split_win_size=8的数据
        split_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
        if not split_data:
            print(f"Warning: {model_name} has no split_win_size={analyz_split_win_size} data for {task_name}")
            continue
            
        # 获取random_score的最小值
        if "chunk_win_size" in split_data:
            random_scores = []
            for chunk_size, score_dict in split_data["chunk_win_size"].items():
                random_scores.append(score_dict["random_score"]["main_score"])
            
            if random_scores:
                random_mins[model_name] = min(random_scores)
            else:
                random_mins[model_name] = 0
        else:
            random_mins[model_name] = 0
    
    # 为每个模型分配独特的marks和line组合
    model_style_map = {}
    for i, model_name in enumerate(all_models_data.keys()):
        marker_idx = i % len(model_styles['markers'])
        linestyle_idx = (i // len(model_styles['markers'])) % len(model_styles['linestyles'])
        
        model_style_map[model_name] = {
            'marker': model_styles['markers'][marker_idx],
            'linestyle': model_styles['linestyles'][linestyle_idx],
            'markersize': 3,
            'linewidth': 1,
            'alpha': 0.1  # 原始数据稍微淡化
        }
    
    # 绘制每个模型的原始数据（保留详细样式）
    for model_name, data in all_models_data.items():
        task_data = data["task_name"][task_name]
        baseline = task_data["defult_score"]
        
        # 获取split_win_size=8的数据
        split_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
        if not split_data:
            continue
            
        # 收集数据
        records = []
        if "chunk_win_size" in split_data:
            for chunk_size, score_dict in split_data["chunk_win_size"].items():
                records.append({
                    "chunk_win_size": int(chunk_size),
                    "head_score": score_dict["head_score"]["main_score"],
                    "end_score": score_dict["end_score"]["main_score"],
                    "random_score": score_dict["random_score"]["main_score"],
                    "sort_score": score_dict["sort_score"]["main_score"]
                })
        
        if not records:
            continue
            
        # 转换为DataFrame并排序
        df = pd.DataFrame(records).sort_values("chunk_win_size")
        
        # 归一化处理
        random_min = random_mins[model_name]
        normalization_range = baseline - random_min if baseline > random_min else 1
        
        # 归一化公式：归一化值 = (原始值 - random_min) / normalization_range
        df["head_score_norm"] = (df["head_score"] - random_min) / normalization_range
        df["end_score_norm"] = (df["end_score"] - random_min) / normalization_range
        df["random_score_norm"] = (df["random_score"] - random_min) / normalization_range
        df["sort_score_norm"] = (df["sort_score"] - random_min) / normalization_range
        
        # 存储归一化数据用于拟合
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            all_normalized_data[score_type].append({
                'chunk_sizes': df["chunk_win_size"].values,
                'scores': df[f"{score_type}_norm"].values,
                'model': model_name
            })
        
        # 获取模型的样式
        style = model_style_map[model_name]
        
        # 绘制原始数据线条（保留详细样式）
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            plt.plot(df["chunk_win_size"], df[f"{score_type}_norm"], 
                    color=score_colors[score_type],
                    marker=style['marker'],
                    linestyle=style['linestyle'],
                    markersize=style['markersize'],
                    linewidth=style['linewidth'],
                    alpha=style['alpha'],
                    label=f"{model_name} - {score_labels[score_type]}")
    
    # 拟合并绘制平均曲线（粗线条显示）
    # 获取所有模型共有的chunk_sizes（取并集）
    all_chunk_sizes = set()
    for score_data in all_normalized_data['head_score']:
        all_chunk_sizes.update(score_data['chunk_sizes'])
    
    if all_chunk_sizes:
        common_chunk_sizes = np.array(sorted(all_chunk_sizes))
        
        # 对每种score类型进行拟合
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            if not all_normalized_data[score_type]:
                continue
                
            # 收集所有数据点用于拟合
            all_x = []
            all_y = []
            
            for model_data in all_normalized_data[score_type]:
                all_x.extend(model_data['chunk_sizes'])
                all_y.extend(model_data['scores'])
            
            # 转换为numpy数组
            all_x = np.array(all_x)
            all_y = np.array(all_y)
            
            # 按chunk_size分组计算平均值和标准差
            unique_sizes = np.unique(all_x)
            means = []
            stds = []
            
            for size in unique_sizes:
                mask = all_x == size
                if np.sum(mask) > 0:
                    means.append(np.mean(all_y[mask]))
                    stds.append(np.std(all_y[mask]))
                else:
                    means.append(np.nan)
                    stds.append(np.nan)
            
            means = np.array(means)
            stds = np.array(stds)
            
            # 移除NaN值
            valid_mask = ~np.isnan(means)
            if np.sum(valid_mask) < 2:
                continue
                
            valid_sizes = unique_sizes[valid_mask]
            valid_means = means[valid_mask]
            valid_stds = stds[valid_mask]
            
            # 使用样条插值进行平滑拟合
            if len(valid_sizes) >= 3:
                # 对x取对数进行插值（因为x轴是对数尺度）
                log_sizes = np.log2(valid_sizes)
                log_common = np.log2(common_chunk_sizes)
                
                # 样条插值
                try:
                    spline = interp1d(log_sizes, valid_means, kind='cubic', 
                                    bounds_error=False, fill_value="extrapolate")
                    fitted_means = spline(log_common)
                    
                    # 计算拟合的标准差（使用最近点的标准差）
                    std_interp = interp1d(log_sizes, valid_stds, kind='linear',
                                        bounds_error=False, fill_value="extrapolate")
                    fitted_stds = std_interp(log_common)
                    
                    # 绘制拟合曲线（粗线条）
                    plt.plot(common_chunk_sizes, fitted_means, 
                            color=score_colors[score_type],
                            linewidth=2,
                            alpha=0.9,
                            label=f"Fitted {score_labels[score_type]}")
                    
                    # 添加置信区间（标准差范围）
                    plt.fill_between(common_chunk_sizes, 
                                   fitted_means - fitted_stds,
                                   fitted_means + fitted_stds,
                                   color=score_colors[score_type], 
                                   alpha=0.05)
                    
                except:
                    # 如果样条插值失败，使用线性插值
                    try:
                        linear_interp = interp1d(log_sizes, valid_means, kind='linear',
                                               bounds_error=False, fill_value="extrapolate")
                        fitted_means = linear_interp(log_common)
                        
                        std_interp = interp1d(log_sizes, valid_stds, kind='linear',
                                            bounds_error=False, fill_value="extrapolate")
                        fitted_stds = std_interp(log_common)
                        
                        plt.plot(common_chunk_sizes, fitted_means, 
                                color=score_colors[score_type],
                                linewidth=2,
                                alpha=0.9,
                                label=f"Fitted {score_labels[score_type]}")
                        
                        plt.fill_between(common_chunk_sizes, 
                                       fitted_means - fitted_stds,
                                       fitted_means + fitted_stds,
                                       color=score_colors[score_type], 
                                       alpha=0.9)
                    except:
                        # 如果插值完全失败，直接绘制原始点
                        plt.plot(valid_sizes, valid_means, 
                                color=score_colors[score_type],
                                linewidth=2,
                                alpha=0.9,
                                label=f"Fitted {score_labels[score_type]}")
                        
                        plt.fill_between(valid_sizes, 
                                       valid_means - valid_stds,
                                       valid_means + valid_stds,
                                       color=score_colors[score_type], 
                                       alpha=0.3)
    
    # 添加统一的基线（70%位置）
    plt.axhline(y=1, color="red", linestyle="--", linewidth=2, alpha=0.8, label="Baseline")
    
    # 设置x轴为log2
    plt.xscale("log", base=2)
    
    # 设置x轴刻度
    if all_chunk_sizes:
        xticks = sorted(all_chunk_sizes)
        plt.xticks(xticks, [f"2^{int(np.log2(x))}" for x in xticks])
    
    plt.title(f"{task_name} - Model Comparison (Detailed + Fitted)", fontsize=16, fontweight='bold')
    plt.xlabel("Dimensional Size (2^n)", fontsize=12)
    plt.ylabel("Normalized Score (0% = Random Min, 100% = Baseline)", fontsize=12)
    
    # 创建两个图例：一个用于Score Types（颜色），一个用于模型（marks+line）
    from matplotlib.lines import Line2D
    
    # 第一个图例：Score Types和拟合曲线（颜色）
    score_legend_elements = []
    for score_type, color in score_colors.items():
        score_legend_elements.append(
            Line2D([0], [0], color=color, lw=2, label=f"Fitted {score_labels[score_type]}")
        )
    
    # 第二个图例：模型（marks+line组合）
    model_legend_elements = []
    for model_name, style in model_style_map.items():
        model_legend_elements.append(
            Line2D([0], [0], color='gray', 
                  marker=style['marker'], 
                  linestyle=style['linestyle'],
                  markersize=8,
                  lw=2, 
                  label=model_name)
        )
    
    # 添加基线到图例
    score_legend_elements.append(
        Line2D([0], [0], color='red', linestyle='--', lw=3, alpha=0.7, label='Baseline')
    )
    
    # 创建图例
    legend1 = plt.legend(handles=score_legend_elements, loc='lower left', 
                        bbox_to_anchor=(0.5, 0), title="Fitted Curves", fontsize=10, framealpha=0.9)
    legend2 = plt.legend(handles=model_legend_elements, loc='lower left', 
                        bbox_to_anchor=(0.74, 0), title="Models", fontsize=10, framealpha=0.9)
    
    # 添加图例到图表
    plt.gca().add_artist(legend1)
    plt.gca().add_artist(legend2)
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # 保存图表
    output_dir = "/home/linkco/exa/Useful-Embedding/data/analyze/comparison_visualizations"
    os.makedirs(output_dir, exist_ok=True)
    
    # 替换文件名中的非法字符
    safe_task_name = task_name.replace("/", "_").replace("\\", "_").replace(":", "_")
    save_path = os.path.join(output_dir, f"{safe_task_name}_detailed_fitted_comparison.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved detailed fitted comparison chart for {task_name} to {save_path}")
    
    plt.show()
    plt.close()

print("All detailed fitted comparison charts generated successfully!")

In [ ]:
# 绘制不同模型的拟合结果（按任务类别分类）
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from matplotlib.colors import LinearSegmentedColormap
from scipy import interpolate
from scipy.interpolate import interp1d

# 任务分类表
task_categories = {
    "Classification": [
        'AmazonCounterfactualClassification',
        'AmazonReviewsClassification', 'Banking77Classification',
        'EmotionClassification', 'ImdbClassification',
        'MTOPDomainClassification', 'MTOPIntentClassification',
        'MassiveIntentClassification', 'MassiveScenarioClassification',
        'ToxicConversationsClassification', 'TweetSentimentExtractionClassification'
    ],
    "Clustering": [
        'BiorxivClusteringS2S',
        'MedrxivClusteringS2S',
        'TwentyNewsgroupsClustering'
    ],
    "PairClassification": [
        'SprintDuplicateQuestions', 'TwitterSemEval2015', 'TwitterURLCorpus'
    ],
    "Reranking": [
        'AskUbuntuDupQuestions', 'SciDocsRR', 'StackOverflowDupQuestions'
    ],
    "STS": [
        'BIOSSES', 'SICK-R', 'STS12', 'STS13', 'STS14', 'STS15',
        'STS16', 'STS17', 'STSBenchmark'
    ],
    "Retrieval": [
        'ArguAna', 'CQADupstackEnglishRetrieval',
        'NFCorpus', 'SCIDOCS', 'SciFact'
    ],
    "MTEB": [
        'AmazonCounterfactualClassification',
        'AmazonReviewsClassification', 'Banking77Classification',
        'EmotionClassification', 'ImdbClassification',
        'MTOPDomainClassification', 'MTOPIntentClassification',
        'MassiveIntentClassification', 'MassiveScenarioClassification',
        'ToxicConversationsClassification', 'TweetSentimentExtractionClassification',
        'BiorxivClusteringS2S',
        'MedrxivClusteringS2S',
        'TwentyNewsgroupsClustering',
        'SprintDuplicateQuestions', 'TwitterSemEval2015', 'TwitterURLCorpus',
        'AskUbuntuDupQuestions', 'SciDocsRR', 'StackOverflowDupQuestions',
        'BIOSSES', 'SICK-R', 'STS12', 'STS13', 'STS14', 'STS15',
        'STS16', 'STS17', 'STSBenchmark',
        'ArguAna', 'CQADupstackEnglishRetrieval',
        'NFCorpus', 'SCIDOCS', 'SciFact'
    ],
}


    # "Summarization": [
    #     'SummEval'
    # ],

# 指定文件夹路径
folder_path = '/home/linkco/exa/Useful-Embedding/data/analyze/'

# 获取所有文件和文件夹名称
files = os.listdir(folder_path)

# 筛选出以.json结尾的文件
json_files = [f for f in files if f.endswith('.json')]

# 选择要比较的模型（可以根据需要修改这个列表）
models_to_compare = ['stella_en_400M_v5-GEDI-epoch_3.json']  # 示例模型名称

# 存储所有模型的数据
all_models_data = {}

# 选择需要分析的最小切割窗口
analyz_split_win_size = 2

# 读取所有模型数据
for file in json_files:
# for file in models_to_compare:
    if file not in models_to_compare:
        model_name = file.replace(".json", "")
        
        print(f"Processing {model_name}")
        
        # 读取文件
        file_path = folder_path + "{}.json".format(model_name)
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        all_models_data[model_name] = data

# 定义Score Types的颜色和标签
score_colors = {
    'head_score': '#1f77b4',    # 蓝色 - Best Chunks
    'end_score': '#ff7f0e',     # 橙色 - Poor Chunks  
    'random_score': '#2ca02c',  # 绿色 - Random Chunks
    # 'sort_score': '#d62728'     # 红色 - Sort Chunks
    'sort_score': '#000000'     # 黑色 - Sort Chunks
}

score_labels = {
    'head_score': 'Best',
    'end_score': 'Poor',
    'random_score': 'Random',
    'sort_score': 'Sort'
}

# 定义模型的marks和line样式组合
model_styles = {
    'markers': ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'X', 'd'],
    'linestyles': ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]
}

# 为每个任务类别创建综合图表
for category_name, category_tasks in task_categories.items():
    print(f"\nProcessing category: {category_name}")
    print(f"Tasks in category: {category_tasks}")
    
    plt.figure(figsize=(6, 4))
    
    # 存储所有模型的归一化数据用于拟合（按类别聚合）
    all_normalized_data = {
        'head_score': [],
        'end_score': [],
        'random_score': [],
        'sort_score': []
    }
    
    # 存储每个模型的random_min值用于归一化（按类别聚合）
    random_mins = {}
    
    # 首先检查该类别下哪些任务实际存在于数据中
    available_tasks = []
    for model_name, data in all_models_data.items():
        model_tasks = list(data["task_name"].keys())
        available_tasks.extend([task for task in category_tasks if task in model_tasks])
    
    available_tasks = list(set(available_tasks))  # 去重
    print(f"Available tasks in data: {available_tasks}")
    
    if not available_tasks:
        print(f"No available tasks found for category {category_name}, skipping...")
        plt.close()
        continue
    
    # 首先收集所有模型的random_min值（按类别平均）
    for model_name, data in all_models_data.items():
        category_random_mins = []
        
        for task_name in available_tasks:
            if task_name not in data["task_name"]:
                continue
                
            task_data = data["task_name"][task_name]
            baseline = task_data["defult_score"]
            
            # 获取split_win_size=8的数据
            split_8_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
            if not split_8_data:
                continue
                
            # 获取random_score的最小值
            if "chunk_win_size" in split_8_data:
                random_scores = []
                for chunk_size, score_dict in split_8_data["chunk_win_size"].items():
                    random_scores.append(score_dict["random_score"]["main_score"])
                
                if random_scores:
                    category_random_mins.append(min(random_scores))
        
        if category_random_mins:
            random_mins[model_name] = np.mean(category_random_mins)
        else:
            random_mins[model_name] = 0
    
    # 为每个模型分配独特的marks和line组合
    model_style_map = {}
    for i, model_name in enumerate(all_models_data.keys()):
        marker_idx = i % len(model_styles['markers'])
        linestyle_idx = (i // len(model_styles['markers'])) % len(model_styles['linestyles'])
        
        model_style_map[model_name] = {
            'marker': model_styles['markers'][marker_idx],
            'linestyle': model_styles['linestyles'][linestyle_idx],
            'markersize': 3,
            'linewidth': 1,
            'alpha': 0.1  # 原始数据稍微淡化
        }
    
    # 绘制每个模型的原始数据（按类别聚合）
    for model_name, data in all_models_data.items():
        # 收集该类别下所有任务的数据
        category_records = []
        
        for task_name in available_tasks:
            if task_name not in data["task_name"]:
                continue
                
            task_data = data["task_name"][task_name]
            baseline = task_data["defult_score"]
            
            # 获取split_win_size=8的数据
            split_8_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
            if not split_8_data:
                continue
                
            # 收集数据
            if "chunk_win_size" in split_8_data:
                for chunk_size, score_dict in split_8_data["chunk_win_size"].items():
                    category_records.append({
                        "chunk_win_size": int(chunk_size),
                        "head_score": score_dict["head_score"]["main_score"],
                        "end_score": score_dict["end_score"]["main_score"],
                        "random_score": score_dict["random_score"]["main_score"],
                        "sort_score": score_dict["sort_score"]["main_score"],
                        "baseline": baseline,
                        "task": task_name
                    })
        
        if not category_records:
            continue
            
        # 转换为DataFrame
        df = pd.DataFrame(category_records)
        
        # 按chunk_win_size分组计算平均值（跨任务平均）
        df_avg = df.groupby("chunk_win_size").agg({
            "head_score": "mean",
            "end_score": "mean", 
            "random_score": "mean",
            "sort_score": "mean",
            "baseline": "mean"
        }).reset_index().sort_values("chunk_win_size")
        
        # 计算平均baseline
        avg_baseline = df["baseline"].mean()
        
        # 归一化处理
        random_min = random_mins[model_name]
        normalization_range = avg_baseline - random_min if avg_baseline > random_min else 1
        
        # 归一化公式：归一化值 = (原始值 - random_min) / normalization_range
        df_avg["head_score_norm"] = (df_avg["head_score"] - random_min) / normalization_range
        df_avg["end_score_norm"] = (df_avg["end_score"] - random_min) / normalization_range
        df_avg["random_score_norm"] = (df_avg["random_score"] - random_min) / normalization_range
        df_avg["sort_score_norm"] = (df_avg["sort_score"] - random_min) / normalization_range
        
        # 存储归一化数据用于拟合
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            all_normalized_data[score_type].append({
                'chunk_sizes': df_avg["chunk_win_size"].values,
                'scores': df_avg[f"{score_type}_norm"].values,
                'model': model_name
            })
        
        # 获取模型的样式
        style = model_style_map[model_name]
        
        # 绘制原始数据线条（保留详细样式）
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            plt.plot(df_avg["chunk_win_size"], df_avg[f"{score_type}_norm"], 
                    color=score_colors[score_type],
                    marker=style['marker'],
                    linestyle=style['linestyle'],
                    markersize=style['markersize'],
                    linewidth=style['linewidth'],
                    alpha=style['alpha'],
                    label=f"{model_name} - {score_labels[score_type]}")
    
    # 拟合并绘制平均曲线（粗线条显示）
    # 获取所有模型共有的chunk_sizes（取并集）
    all_chunk_sizes = set()
    for score_data in all_normalized_data['head_score']:
        all_chunk_sizes.update(score_data['chunk_sizes'])
    
    if all_chunk_sizes:
        common_chunk_sizes = np.array(sorted(all_chunk_sizes))
        
        # 对每种score类型进行拟合
        for score_type in ['head_score', 'end_score', 'random_score', 'sort_score']:
            if not all_normalized_data[score_type]:
                continue
                
            # 收集所有数据点用于拟合
            all_x = []
            all_y = []
            
            for model_data in all_normalized_data[score_type]:
                all_x.extend(model_data['chunk_sizes'])
                all_y.extend(model_data['scores'])
            
            # 转换为numpy数组
            all_x = np.array(all_x)
            all_y = np.array(all_y)
            
            # 按chunk_size分组计算平均值和标准差
            unique_sizes = np.unique(all_x)
            means = []
            stds = []
            
            for size in unique_sizes:
                mask = all_x == size
                if np.sum(mask) > 0:
                    means.append(np.mean(all_y[mask]))
                    stds.append(np.std(all_y[mask]))
                else:
                    means.append(np.nan)
                    stds.append(np.nan)
            
            means = np.array(means)
            stds = np.array(stds)
            
            # 移除NaN值
            valid_mask = ~np.isnan(means)
            if np.sum(valid_mask) < 2:
                continue
                
            valid_sizes = unique_sizes[valid_mask]
            valid_means = means[valid_mask]
            valid_stds = stds[valid_mask]
            
            # 使用样条插值进行平滑拟合
            if len(valid_sizes) >= 3:
                # 对x取对数进行插值（因为x轴是对数尺度）
                log_sizes = np.log2(valid_sizes)
                log_common = np.log2(common_chunk_sizes)
                
                # 样条插值
                try:
                    spline = interp1d(log_sizes, valid_means, kind='cubic', 
                                    bounds_error=False, fill_value="extrapolate")
                    fitted_means = spline(log_common)
                    
                    # 计算拟合的标准差（使用最近点的标准差）
                    std_interp = interp1d(log_sizes, valid_stds, kind='linear',
                                        bounds_error=False, fill_value="extrapolate")
                    fitted_stds = std_interp(log_common)
                    
                    # 绘制拟合曲线（粗线条）
                    plt.plot(common_chunk_sizes, fitted_means, 
                            color=score_colors[score_type],
                            linewidth=3,
                            alpha=0.9,
                            label=f"{score_labels[score_type]}")
                    
                    # 添加置信区间（标准差范围）
                    plt.fill_between(common_chunk_sizes, 
                                   fitted_means - fitted_stds,
                                   fitted_means + fitted_stds,
                                   color=score_colors[score_type], 
                                   alpha=0.1)
                    
                except:
                    # 如果样条插值失败，使用线性插值
                    try:
                        linear_interp = interp1d(log_sizes, valid_means, kind='linear',
                                               bounds_error=False, fill_value="extrapolate")
                        fitted_means = linear_interp(log_common)
                        
                        std_interp = interp1d(log_sizes, valid_stds, kind='linear',
                                            bounds_error=False, fill_value="extrapolate")
                        fitted_stds = std_interp(log_common)
                        
                        plt.plot(common_chunk_sizes, fitted_means, 
                                color=score_colors[score_type],
                                linewidth=3,
                                alpha=0.9,
                                label=f"{score_labels[score_type]}")
                        
                        plt.fill_between(common_chunk_sizes, 
                                       fitted_means - fitted_stds,
                                       fitted_means + fitted_stds,
                                       color=score_colors[score_type], 
                                       alpha=0.1)
                    except:
                        # 如果插值完全失败，直接绘制原始点
                        plt.plot(valid_sizes, valid_means, 
                                color=score_colors[score_type],
                                linewidth=3,
                                alpha=0.9,
                                label=f"{score_labels[score_type]}")
                        
                        plt.fill_between(valid_sizes, 
                                       valid_means - valid_stds,
                                       valid_means + valid_stds,
                                       color=score_colors[score_type], 
                                       alpha=0.2)
    
    # 添加统一的基线（100%位置）
    plt.axhline(y=1, color="red", linestyle="--", linewidth=2, alpha=0.8, label="Baseline")
    
    # # 设置x轴为log2
    # plt.xscale("log", base=2)
    
    # 设置x轴刻度
    if all_chunk_sizes:
        # xticks = sorted(all_chunk_sizes)
        xticks = sorted([2, 64, 128, 256, 384, 512, 768, 1024])
        # plt.xticks(xticks, [f"2^{int(np.log2(x))}" for x in xticks])
        plt.xticks(xticks, xticks)
    
    plt.title(f"{category_name}", 
              fontsize=12, fontweight='bold')
    plt.xlabel("Dimensional Size", fontsize=12)
    plt.ylabel("Normalized Score", fontsize=10)
    
    # 创建两个图例：一个用于Score Types（颜色），一个用于模型（marks+line）
    from matplotlib.lines import Line2D
    
    # 第一个图例：Score Types和拟合曲线（颜色）
    score_legend_elements = []
    for score_type, color in score_colors.items():
        score_legend_elements.append(
            Line2D([0], [0], color=color, lw=2, label=f"{score_labels[score_type]}")
        )
    
    # 第二个图例：模型（marks+line组合）
    model_legend_elements = []
    for model_name, style in model_style_map.items():
        model_legend_elements.append(
            Line2D([0], [0], color='gray', 
                  marker=style['marker'], 
                  linestyle=style['linestyle'],
                  markersize=6,
                  lw=2, 
                  label=model_name)
        )
    
    # 添加基线到图例
    score_legend_elements.append(
        Line2D([0], [0], color='red', linestyle='--', lw=3, alpha=0.7, label='Baseline')
    )
    
    # 创建图例
    legend1 = plt.legend(handles=score_legend_elements, loc='lower left', 
                        bbox_to_anchor=(0.58, 0.02), title="Types", fontsize=6, framealpha=0.6)
    legend2 = plt.legend(handles=model_legend_elements, loc='lower left', 
                        bbox_to_anchor=(0.72, 0.02), title="Models", fontsize=6, framealpha=0.6)

    # 添加图例到图表
    plt.gca().add_artist(legend1)
    plt.gca().add_artist(legend2)
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # 保存图表
    output_dir = "/home/linkco/exa/Useful-Embedding/data/analyze/category_comparison_visualizations"
    os.makedirs(output_dir, exist_ok=True)
    
    # 替换文件名中的非法字符
    safe_category_name = category_name.replace("/", "_").replace("\\", "_").replace(":", "_")
    save_path = os.path.join(output_dir, f"{safe_category_name}_category_comparison.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved category comparison chart for {category_name} to {save_path}")
    
    plt.show()
    plt.close()

print("All category comparison charts generated successfully!")

In [ ]:
# 绘制不同模型的拟合结果（按任务类别分类）——2×3总图 + 每子图独立图例
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from scipy.interpolate import interp1d
from matplotlib.lines import Line2D

# ===============================
# 任务分类定义
# ===============================
task_categories = {
    "Classification": [
        'AmazonCounterfactualClassification', 'AmazonReviewsClassification', 'Banking77Classification',
        'EmotionClassification', 'ImdbClassification', 'MTOPDomainClassification', 'MTOPIntentClassification',
        'MassiveIntentClassification', 'MassiveScenarioClassification', 'ToxicConversationsClassification',
        'TweetSentimentExtractionClassification'
    ],
    "Clustering": ['BiorxivClusteringS2S', 'MedrxivClusteringS2S', 'TwentyNewsgroupsClustering'],
    "PairClassification": ['SprintDuplicateQuestions', 'TwitterSemEval2015', 'TwitterURLCorpus'],
    "Reranking": ['AskUbuntuDupQuestions', 'SciDocsRR', 'StackOverflowDupQuestions'],
    "STS": ['BIOSSES', 'SICK-R', 'STS12', 'STS13', 'STS14', 'STS15', 'STS16', 'STS17', 'STSBenchmark'],
    "Retrieval": ['ArguAna', 'CQADupstackEnglishRetrieval', 'NFCorpus', 'SCIDOCS', 'SciFact']
}

# ===============================
# 文件读取
# ===============================
folder_path = '/home/linkco/exa/Useful-Embedding/data/analyze/'
json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
models_to_compare = ['stella_en_400M_v5-GEDI-epoch_3.json']  # 修改为你要比较的模型

all_models_data = {}
analyz_split_win_size = 2

for file in json_files:
    if file not in models_to_compare:
        model_name = file.replace(".json", "")
        print(f"Processing {model_name}")
        with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
            data = json.load(f)
        all_models_data[model_name] = data

# ===============================
# 绘图样式定义
# ===============================
score_colors = {
    'head_score': '#1f77b4',
    'end_score': '#ff7f0e',
    'random_score': '#2ca02c',
    # 'sort_score': '#d62728'     # 红色 - Sort Chunks
    'sort_score': '#000000'     # 黑色 - Sort Chunks
}
score_labels = {
    'head_score': 'Best',
    'end_score': 'Poor',
    'random_score': 'Random',
    'sort_score': 'Sort'
}
model_styles = {
    'markers': ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'X', 'd'],
    'linestyles': ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]
}

# ===============================
# 创建2×3图布局
# ===============================
selected_categories = ["Classification", "Clustering", "PairClassification",
                       "Reranking", "STS", "Retrieval"]

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
axes = axes.flatten()
fig.subplots_adjust(wspace=0.3, hspace=0.4)

# ===============================
# 主循环：每个类别一个子图
# ===============================
for cat_idx, category_name in enumerate(selected_categories):
    category_tasks = task_categories[category_name]
    ax = axes[cat_idx]
    print(f"\nProcessing category: {category_name}")

    all_normalized_data = {k: [] for k in score_colors.keys()}
    random_mins = {}

    # 检查可用任务
    available_tasks = []
    for model_name, data in all_models_data.items():
        model_tasks = list(data["task_name"].keys())
        available_tasks.extend([t for t in category_tasks if t in model_tasks])
    available_tasks = list(set(available_tasks))
    if not available_tasks:
        ax.set_title(category_name)
        continue

    # 计算random_min
    for model_name, data in all_models_data.items():
        category_random_mins = []
        for task_name in available_tasks:
            if task_name not in data["task_name"]:
                continue
            task_data = data["task_name"][task_name]
            split_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
            if not split_data or "chunk_win_size" not in split_data:
                continue
            random_scores = [v["random_score"]["main_score"] for v in split_data["chunk_win_size"].values()]
            if random_scores:
                category_random_mins.append(min(random_scores))
        random_mins[model_name] = np.mean(category_random_mins) if category_random_mins else 0

    # 分配模型样式
    model_style_map = {}
    for i, model_name in enumerate(all_models_data.keys()):
        model_style_map[model_name] = {
            'marker': model_styles['markers'][i % len(model_styles['markers'])],
            'linestyle': model_styles['linestyles'][(i // len(model_styles['markers'])) % len(model_styles['linestyles'])],
            'markersize': 3, 'linewidth': 1, 'alpha': 0.15
        }

    # 绘制原始曲线
    for model_name, data in all_models_data.items():
        category_records = []
        for task_name in available_tasks:
            if task_name not in data["task_name"]:
                continue
            task_data = data["task_name"][task_name]
            baseline = task_data["defult_score"]
            split_data = task_data["split_win_size"].get(str(analyz_split_win_size), {})
            if not split_data or "chunk_win_size" not in split_data:
                continue
            for chunk_size, score_dict in split_data["chunk_win_size"].items():
                category_records.append({
                    "chunk_win_size": int(chunk_size),
                    **{k: score_dict[k]["main_score"] for k in score_colors.keys()},
                    "baseline": baseline
                })
        if not category_records:
            continue
        df = pd.DataFrame(category_records)
        df_avg = df.groupby("chunk_win_size").mean().reset_index().sort_values("chunk_win_size")

        avg_baseline = df_avg["baseline"].mean()
        random_min = random_mins[model_name]
        normalization_range = avg_baseline - random_min if avg_baseline > random_min else 1
        for s in score_colors.keys():
            df_avg[f"{s}_norm"] = (df_avg[s] - random_min) / normalization_range
            all_normalized_data[s].append({
                'chunk_sizes': df_avg["chunk_win_size"].values,
                'scores': df_avg[f"{s}_norm"].values
            })
        style = model_style_map[model_name]
        for s in score_colors.keys():
            ax.plot(df_avg["chunk_win_size"], df_avg[f"{s}_norm"],
                    color=score_colors[s],
                    marker=style['marker'], linestyle=style['linestyle'],
                    markersize=style['markersize'], linewidth=style['linewidth'],
                    alpha=style['alpha'])

    # 拟合平均曲线
    all_chunk_sizes = sorted({x for d in all_normalized_data['head_score'] for x in d['chunk_sizes']})
    if all_chunk_sizes:
        common_chunk_sizes = np.array(all_chunk_sizes)
        for s in score_colors.keys():
            if not all_normalized_data[s]:
                continue
            all_x, all_y = [], []
            for d in all_normalized_data[s]:
                all_x.extend(d['chunk_sizes'])
                all_y.extend(d['scores'])
            all_x, all_y = np.array(all_x), np.array(all_y)
            unique_sizes = np.unique(all_x)
            means, stds = [], []
            for size in unique_sizes:
                mask = all_x == size
                means.append(np.mean(all_y[mask]))
                stds.append(np.std(all_y[mask]))
            valid = ~np.isnan(means)
            if np.sum(valid) < 2:
                continue
            x, y, ystd = unique_sizes[valid], np.array(means)[valid], np.array(stds)[valid]
            try:
                f = interp1d(np.log2(x), y, kind='cubic', fill_value="extrapolate")
                f_std = interp1d(np.log2(x), ystd, kind='linear', fill_value="extrapolate")
                x_common = np.log2(common_chunk_sizes)
                y_fit, y_fit_std = f(x_common), f_std(x_common)
                ax.plot(common_chunk_sizes, y_fit, color=score_colors[s], lw=2.5, alpha=0.9, label=score_labels[s])
                ax.fill_between(common_chunk_sizes, y_fit - y_fit_std, y_fit + y_fit_std,
                                color=score_colors[s], alpha=0.1)
            except:
                ax.plot(x, y, color=score_colors[s], lw=2.5, alpha=0.9, label=score_labels[s])
    ax.axhline(y=1, color="red", linestyle="--", lw=1.5, alpha=0.8, label="Baseline")

    # 坐标与标题
    ax.set_title(category_name, fontsize=10, fontweight='bold')
    # ax.set_xlabel("Dimensional Size", fontsize=9)
    ax.set_xticks([2, 64, 128, 256, 384, 512, 768, 1024])
    ax.grid(True, alpha=0.3)

    # ============ 每子图图例 ============
    # Score legend（颜色）
    score_legend_elements = [
        Line2D([0], [0], color=c, lw=2, label=l)
        for c, l in zip(score_colors.values(), score_labels.values())
    ] + [Line2D([0], [0], color='red', linestyle='--', lw=2, label='Baseline')]

    # Model legend（marker）
    model_legend_elements = []
    for model_name, style in model_style_map.items():
        model_legend_elements.append(Line2D(
            [0], [0], color='gray',
            marker=style['marker'], linestyle=style['linestyle'],
            markersize=6, lw=1.5, label=model_name
        ))

    # 添加两个图例
    legend1 = ax.legend(handles=score_legend_elements, loc='lower left',
                        bbox_to_anchor=(0.45, 0.02), fontsize=6, framealpha=0.6, title="Types")
    legend2 = ax.legend(handles=model_legend_elements, loc='lower left',
                        bbox_to_anchor=(0.64, 0.02), fontsize=6, framealpha=0.6, title="Models")
    ax.add_artist(legend1)

# ===============================
# 保存
# ===============================
plt.tight_layout()
output_dir = "/home/linkco/exa/Useful-Embedding/data/analyze/category_comparison_visualizations"
os.makedirs(output_dir, exist_ok=True)
save_path = os.path.join(output_dir, "AllCategories_2x3_with_legends.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved combined 2×3 comparison chart with legends to {save_path}")


In [ ]:
a = [5,2,4,9,1,43,6,8,2,43,6,6,2,76]
sorted_index_list = [index for index, value in sorted(enumerate(a), key=lambda x: x[1], reverse=True)]
print(sorted_index_list)